# Single Cell Analysis Data Visualization with Napari

This notebook provides interactive, GUI-based visualization tools for single-cell data using [napari](https://napari.org), including:
- Multi-wavelength visualization
- Label/segmentation overlay visualization
- 3D TIFF data exploration
- Interactive controls for visualization parameters

Napari is a multi-dimensional image viewer for Python that's designed for browsing, annotating, and analyzing large multi-dimensional images. It's particularly well-suited for scientific imaging data like the single-cell analysis we're working with.

> **Scope note:** this notebook needs a real display and the `gui` extra (`uv sync --extra gui`) — it opens an interactive napari window. For headless/HPC use (no display, matplotlib-based), see `00b_headless_data_visualization.ipynb` instead.

## 1. Import Dependencies and Custom Modules

In [ ]:
from pathlib import Path

import napari
import tifffile

# Import interactive widgets
import ipywidgets as widgets
from IPython.display import display

# Reuse the existing napari/display helpers instead of redefining loading and
# viewer-construction logic here (see src/visualize/visualize_prediction.py and
# src/visualize/headless_layers.py)
from src.visualize import create_enhanced_napari_viewer, normalize_for_display

# Enable Napari in notebook
%gui qt

## 2. Notebook Constants & Data Paths

In [ ]:
# Root dir for the 3D-TIFF example data used throughout this notebook.
# `data/` is not tracked in git, so adjust this to a 3D dataset available on your
# machine (e.g. the output of `scripts/run_postprocessing.py` with 3D stacking enabled).
RAW_3D_DIR = Path("data/sample_data_output/3d_images")

bf_path = next(RAW_3D_DIR.glob("*_BF_3d.tif"))
mask_path = bf_path.with_name(bf_path.name.replace("_BF_3d.tif", "_Cells_3d.tif"))
mask_path = mask_path if mask_path.exists() else None

print(f"Brightfield: {bf_path}")
print(f"Labels:      {mask_path}")

## 3. Visualize Multi-Wavelength Data

In [ ]:
def visualize_wavelengths(data, channel_names=None):
    """Visualize multi-wavelength data in napari with custom colormaps"""
    viewer = napari.Viewer()
    
    # Define some standard colormaps for common wavelengths
    colormaps = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow']
    
    # Add each wavelength as a separate layer
    num_channels = data.shape[0] if data.ndim > 2 else 1
    
    for i in range(num_channels):
        # Get channel data
        channel_data = data[i] if data.ndim > 2 else data
        
        # Get channel name
        if channel_names and i < len(channel_names):
            name = channel_names[i]
        else:
            name = f'Wavelength {i}'
        
        # Add to viewer with appropriate colormap
        viewer.add_image(
            channel_data,
            name=name,
            colormap=colormaps[i % len(colormaps)],
            blending='additive'
        )
    
    return viewer

## 4. Interactive Visualization with Widgets

In [ ]:
def create_interactive_napari_viewer(data, labels=None, viewer=None):
    """Create an interactive napari viewer with widgets to control visualization"""
    if viewer is None:
        viewer = napari.Viewer()

    # Check if data is multichannel
    if data.ndim > 2 and data.shape[0] < 10:
        num_channels = data.shape[0]
        channel_layers = []
        
        # Add each channel with visibility control
        for i in range(num_channels):
            layer = viewer.add_image(
                data[i],
                name=f'Channel {i}',
                colormap=['red', 'green', 'blue', 'cyan', 'magenta'][i % 5],
                blending='additive',
                visible=(i == 0)  # Only first channel visible initially
            )
            channel_layers.append(layer)
        
        # Add labels if provided
        if labels is not None:
            viewer.add_labels(
                labels,
                name='Segmentation',
                opacity=0.5
            )
        
        # Create visibility control widgets
        channel_checkboxes = []
        for i in range(num_channels):
            checkbox = widgets.Checkbox(
                value=(i == 0),
                description=f'Channel {i}',
                layout=widgets.Layout(width='150px')
            )
            
            # Link checkbox to layer visibility
            def make_toggle_func(layer):
                def toggle_visibility(change):
                    layer.visible = change['new']
                return toggle_visibility
            
            checkbox.observe(make_toggle_func(channel_layers[i]), names='value')
            channel_checkboxes.append(checkbox)
        
        # Create widget for labels opacity
        if labels is not None:
            opacity_slider = widgets.FloatSlider(
                value=0.5,
                min=0,
                max=1.0,
                step=0.05,
                description='Labels Opacity:',
                layout=widgets.Layout(width='250px')
            )
            
            def update_opacity(change):
                viewer.layers['Segmentation'].opacity = change['new']
                
            opacity_slider.observe(update_opacity, names='value')
            
            # Display widgets
            display(widgets.VBox([
                widgets.HBox(channel_checkboxes),
                opacity_slider
            ]))
        else:
            display(widgets.HBox(channel_checkboxes))
    
    else:
        # For non-multichannel data
        viewer.add_image(data, name='Image')
        
        if labels is not None:
            viewer.add_labels(labels, name='Segmentation', opacity=0.5)
    
    return viewer

## 5. Example Usage

### Load and visualize brightfield data

```python
# bf_path was resolved in Section 2 from RAW_3D_DIR
viewer = create_enhanced_napari_viewer(brightfield=bf_path)
```

### Visualize multi-wavelength data

Illustrative only — the current mCherry-focused datasets are single/dual-channel, not
multi-wavelength fluorescence, so there's no runnable sample path wired up for this one.

```python
channel_names = ['DAPI', 'GFP', 'RFP']
data = tifffile.imread('path/to/multi_wavelength.tif')  # shape (channels, Y, X)

# Visualize multi-wavelength data
viewer = visualize_wavelengths(data, channel_names)
```

### Visualize image with labels overlay

```python
# bf_path / mask_path were resolved in Section 2
viewer = create_enhanced_napari_viewer(brightfield=bf_path, ground_truth=mask_path)
```

### Interactive visualization

Note: `create_interactive_napari_viewer`'s channel-vs-labels heuristic assumes
channel-first data with fewer than 10 channels (`data.shape[0] < 10`) — a 3D BF volume
with a small z-extent could be misread as multichannel. Check your data's shape first.

```python
bf = tifffile.imread(bf_path)
bf_norm = normalize_for_display(bf)
labels = tifffile.imread(mask_path) if mask_path is not None else None

# Create interactive viewer with channel selection
viewer = create_interactive_napari_viewer(bf_norm, labels)
```

In [ ]:
# Basic visualization: brightfield + labels overlay, using the paths resolved in
# Section 2 (RAW_3D_DIR / bf_path / mask_path)
viewer = create_enhanced_napari_viewer(brightfield=bf_path, ground_truth=mask_path)

In [ ]:
viewer.show()

## 6. Advanced Features

In [ ]:
def create_3d_visualization(data, labels=None):
    """Create a 3D visualization of volumetric data"""
    viewer = napari.Viewer(ndisplay=3)  # Set to 3D mode
    
    # Add the data
    viewer.add_image(
        data,
        name='Volume',
        colormap='gray',
        rendering='mip'  # Maximum intensity projection
    )
    
    # Add labels if provided
    if labels is not None:
        viewer.add_labels(
            labels,
            name='Segmentation',
            opacity=0.3
        )
    
    return viewer

def create_time_series_visualization(data):
    """Visualize time-series data with napari"""
    viewer = napari.Viewer()
    
    # Add the time series data
    viewer.add_image(
        data,
        name='Time Series',
        colormap='viridis'
    )
    
    return viewer